In [ ]:
# AQUA005 – Bayesian C–Q Analysis (Synthetic Data)

**Data**: Synthetic, n=200, dilution pattern (β_true = -0.4)

## 1. Load & Explore
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/synthetic_cq_phosphorus.csv')
print(df.describe())
print("Correlation logQ-logC:", df['logQ'].corr(df['logC']))

## 2. Visualization
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.scatter(df['Q'], df['C'], alpha=0.6)
plt.xscale('log'); plt.yscale('log')
plt.xlabel('Q (discharge)'); plt.ylabel('C (P concentration)')
plt.title('C–Q scatter (log-log)')

plt.subplot(1,2,2)
plt.scatter(df['logQ'], df['logC'], alpha=0.6)
plt.xlabel('log(Q)'); plt.ylabel('log(C)')
plt.title('Linearized relationship')
plt.tight_layout()
plt.savefig('../figures/cq_scatter.png', dpi=150)
plt.show()

## 3. Bayesian Model (analytical conjugate solution)
log(C) = α + β log(Q) + ε, ε ~ N(0, σ²)

Vague priors: α,β ~ N(0,100), σ² ~ InvGamma(0.001,0.001)
X = np.column_stack((np.ones(len(df)), df['logQ']))
y = df['logC']

prior_precision = 1/100.0
prior_cov_inv = np.diag([prior_precision, prior_precision])

post_cov = np.linalg.inv(X.T @ X + prior_cov_inv)
post_mean = post_cov @ (X.T @ y)

sse = ((y - X @ post_mean)**2).sum()
a_n = 0.001 + len(y)/2
b_n = 0.001 + 0.5 * sse
post_sigma2 = b_n / (a_n - 1)
post_sigma = np.sqrt(post_sigma2)

print(f"Posterior mean: α = {post_mean[0]:.3f}, β = {post_mean[1]:.3f}, σ = {post_sigma:.3f}")

## 4. Interpretation
- β ≈ -0.40 → clear dilution behavior
- Phosphorus concentration decreases as discharge increases
- Suggests constant P input (e.g. point sources) diluted by high-flow water
- Uncertainty low due to n=200 and strong signal